# ETL Silver → Gold

Popula o Star Schema no PostgreSQL a partir dos dados limpos do Silver Layer.

## 1. Imports

Bibliotecas necessarias para o ETL.

In [131]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
import warnings
from sqlalchemy import create_engine

warnings.filterwarnings('ignore', message='.*SQLAlchemy.*')

## 2. Configuracao

Parametros de conexao ao banco de dados.

In [132]:
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'uber_analytics',
    'user': 'postgres',
    'password': 'postgres'
}

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

print('Conexao estabelecida com sucesso!')

Conexao estabelecida com sucesso!


## 3. Verificacao Rapida

Verificar se existem dados no Silver.

In [133]:
cur.execute("SELECT COUNT(*) FROM silver.booking")
silver_total = cur.fetchone()[0]
print(f'Total de registros no Silver: {silver_total:,}')

if silver_total == 0:
    print('ALERTA: Nao ha dados no Silver. Execute o ETL raw_to_silver primeiro.')
else:
    print(f'Silver OK: {silver_total:,} registros encontrados.')

Total de registros no Silver: 148,767
Silver OK: 148,767 registros encontrados.


## 4. Executar DDL

Criar schema DW e tabelas (dimensoes + fato).

In [134]:
import os

ddl_path = os.path.join('..', 'Data Layer', 'gold', 'ddl.sql')

with open(ddl_path, 'r', encoding='utf-8') as f:
    ddl_sql = f.read()

cur.execute(ddl_sql)
conn.commit()

print('DDL executado com sucesso! Schema DW criado.')

DDL executado com sucesso! Schema DW criado.


## 5. Carregar Dados do Silver

Ler todos os dados da tabela silver.booking.

In [135]:
query = "SELECT * FROM silver.booking"
df_silver = pd.read_sql(query, conn)

print(f'Dados carregados: {len(df_silver):,} registros')
print(f'Colunas: {list(df_silver.columns)}')

Dados carregados: 148,767 registros
Colunas: ['id', 'date', 'time', 'status', 'customer_id', 'vehicle', 'pickup', 'drop', 'pickup_zone', 'pickup_region', 'drop_zone', 'drop_region', 'vtat', 'ctat', 'cancelled_by_customer', 'reason_cancelled_by_customer', 'cancelled_by_driver', 'reason_cancelled_by_driver', 'incomplete', 'reason_incomplete', 'value', 'distance', 'distance_category', 'value_per_km', 'driver_rating', 'customer_rating', 'payment']


## 6. Dimensao Tempo (dim_tmp)

Criar dimensao de tempo com hierarquia completa.

In [136]:
df_silver['date'] = pd.to_datetime(df_silver['date'])

dim_tmp = df_silver[['date']].drop_duplicates().copy()
dim_tmp.rename(columns={'date': 'dat'}, inplace=True)

dim_tmp['ano'] = dim_tmp['dat'].dt.year
dim_tmp['mes'] = dim_tmp['dat'].dt.month
dim_tmp['dia'] = dim_tmp['dat'].dt.day
dim_tmp['dia_smn'] = dim_tmp['dat'].dt.dayofweek
dim_tmp['nme_dia_smn'] = dim_tmp['dat'].dt.day_name()
dim_tmp['trm'] = dim_tmp['dat'].dt.quarter
dim_tmp['smn_ano'] = dim_tmp['dat'].dt.isocalendar().week
dim_tmp['ver_fim_smn'] = dim_tmp['dia_smn'].isin([5, 6])
dim_tmp['mes_ano'] = dim_tmp['dat'].dt.strftime('%Y-%m')
dim_tmp['ano_trm'] = dim_tmp['ano'].astype(str) + '-Q' + dim_tmp['trm'].astype(str)

def get_periodo_dia(hora):
    if 6 <= hora < 12:
        return 'Manha'
    elif 12 <= hora < 18:
        return 'Tarde'
    elif 18 <= hora < 24:
        return 'Noite'
    else:
        return 'Madrugada'

dim_tmp['prd_dia'] = df_silver.groupby(df_silver['date'].dt.date)['time'].first().apply(lambda x: get_periodo_dia((x.hour if hasattr(x, 'hour') else int(str(x).split(':')[0])) if pd.notna(x) else 12)).values

def ver_horario_pico(hora):
    return (7 <= hora < 10) or (17 <= hora < 20)

dim_tmp['ver_hrr_pic'] = df_silver.groupby(df_silver['date'].dt.date)['time'].first().apply(lambda x: ver_horario_pico((x.hour if hasattr(x, 'hour') else int(str(x).split(':')[0])) if pd.notna(x) else 12)).values

print(f'Dimensao Tempo: {len(dim_tmp)} registros')

Dimensao Tempo: 365 registros


## 7. Dimensao Localizacao (dim_loc)

Criar dimensao de localizacao com origem e destino.

In [137]:
pickup_locs = df_silver[['pickup', 'pickup_zone', 'pickup_region']].copy()
pickup_locs.columns = ['lcl', 'zna', 'reg']

drop_locs = df_silver[['drop', 'drop_zone', 'drop_region']].copy()
drop_locs.columns = ['lcl', 'zna', 'reg']

dim_loc = pd.concat([pickup_locs, drop_locs], ignore_index=True).drop_duplicates()

def get_tipo_area(zona):
    if 'Airport' in zona:
        return 'Aeroporto'
    elif any(x in zona for x in ['Central', 'Connaught', 'Market']):
        return 'Comercial'
    else:
        return 'Residencial'

dim_loc['tpo_ara'] = dim_loc['zna'].apply(get_tipo_area)

print(f'Dimensao Localizacao: {len(dim_loc)} registros')

Dimensao Localizacao: 176 registros


## 8. Dimensao Veiculo (dim_vei)

Criar dimensao de veiculo com categorias.

In [138]:
dim_vei = df_silver[['vehicle']].drop_duplicates().copy()
dim_vei.rename(columns={'vehicle': 'tpo_vei'}, inplace=True)

def get_categoria_veiculo(tipo):
    tipo_lower = tipo.lower()
    if any(x in tipo_lower for x in ['go', 'mini', 'auto']):
        return 'Economy'
    elif any(x in tipo_lower for x in ['xl', 'suv', 'premier']):
        return 'Premium'
    elif any(x in tipo_lower for x in ['lux', 'black']):
        return 'Luxury'
    else:
        return 'Standard'

dim_vei['ctg_vei'] = dim_vei['tpo_vei'].apply(get_categoria_veiculo)

print(f'Dimensao Veiculo: {len(dim_vei)} registros')

Dimensao Veiculo: 7 registros


## 9. Dimensao Cliente (dim_cli)

Criar dimensao de cliente com segmentacao.

In [139]:
cliente_stats = df_silver.groupby('customer_id').agg(
    ttl_vig_hst=('id', 'count')
).reset_index()

def get_segmento_cliente(total_viagens):
    if total_viagens < 5:
        return 'Occasional'
    elif total_viagens < 20:
        return 'Regular'
    else:
        return 'VIP'

cliente_stats['sgm_cli'] = cliente_stats['ttl_vig_hst'].apply(get_segmento_cliente)

dim_cli = cliente_stats.copy()
dim_cli.rename(columns={'customer_id': 'id_cli'}, inplace=True)

print(f'Dimensao Cliente: {len(dim_cli)} registros')

Dimensao Cliente: 147580 registros


## 10. Dimensao Pagamento (dim_pag)

Criar dimensao de pagamento com categorias.

In [140]:
def get_categoria_pagamento(metodo):
    if pd.isna(metodo) or not isinstance(metodo, str):
        return 'Outros'
    
    metodo = str(metodo).lower()
    
    if metodo == 'cash':
        return 'Cash'
    else:
        return 'Digital'

dim_pag = df_silver[['payment']].dropna().drop_duplicates().copy()
dim_pag = dim_pag.rename(columns={'payment': 'mtd_pag'})
dim_pag['ctg_pag'] = dim_pag['mtd_pag'].apply(get_categoria_pagamento)

print(f'dim_pag preparada: {len(dim_pag)} registros')

dim_pag preparada: 5 registros


## 11. Inserir Dimensoes no Banco

Carregar todas as dimensoes no schema DW.

In [141]:
engine = create_engine(f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

dim_tmp.to_sql('dim_tmp', engine, schema='dw', if_exists='append', index=False)
print(f'dim_tmp: {len(dim_tmp)} registros inseridos')

dim_loc.to_sql('dim_loc', engine, schema='dw', if_exists='append', index=False)
print(f'dim_loc: {len(dim_loc)} registros inseridos')

dim_vei.to_sql('dim_vei', engine, schema='dw', if_exists='append', index=False)
print(f'dim_vei: {len(dim_vei)} registros inseridos')

dim_cli.to_sql('dim_cli', engine, schema='dw', if_exists='append', index=False)
print(f'dim_cli: {len(dim_cli)} registros inseridos')

dim_pag.to_sql('dim_pag', engine, schema='dw', if_exists='append', index=False)
print(f'dim_pag: {len(dim_pag)} registros inseridos')

print('\nTodas as dimensoes foram carregadas com sucesso!')

dim_tmp: 365 registros inseridos
dim_loc: 176 registros inseridos
dim_vei: 7 registros inseridos
dim_cli: 147580 registros inseridos
dim_pag: 5 registros inseridos

Todas as dimensoes foram carregadas com sucesso!


## 12. Recarregar Dimensoes com Chaves

Ler dimensoes do banco para obter as chaves surrogate.

In [142]:
dim_tmp_db = pd.read_sql('SELECT * FROM dw.dim_tmp', conn)
dim_loc_db = pd.read_sql('SELECT * FROM dw.dim_loc', conn)
dim_vei_db = pd.read_sql('SELECT * FROM dw.dim_vei', conn)
dim_cli_db = pd.read_sql('SELECT * FROM dw.dim_cli', conn)
dim_pag_db = pd.read_sql('SELECT * FROM dw.dim_pag', conn)

print('Dimensoes recarregadas com chaves surrogate.')

Dimensoes recarregadas com chaves surrogate.


## 13. Criar Fato Corridas (fat_crr)

Montar tabela fato com FKs para dimensoes.

In [143]:
fat_crr = df_silver.copy()

fat_crr['date'] = pd.to_datetime(fat_crr['date'])
dim_tmp_db['dat'] = pd.to_datetime(dim_tmp_db['dat'])

fat_crr = fat_crr.merge(
    dim_tmp_db[['srk_tmp', 'dat']],
    left_on='date',
    right_on='dat',
    how='left'
).drop('dat', axis=1)

fat_crr = fat_crr.merge(
    dim_loc_db[['srk_loc', 'lcl']],
    left_on='pickup',
    right_on='lcl',
    how='left'
).rename(columns={'srk_loc': 'srk_loc_ori'}).drop('lcl', axis=1)

fat_crr = fat_crr.merge(
    dim_loc_db[['srk_loc', 'lcl']],
    left_on='drop',
    right_on='lcl',
    how='left'
).rename(columns={'srk_loc': 'srk_loc_dst'}).drop('lcl', axis=1)

fat_crr = fat_crr.merge(
    dim_vei_db[['srk_vei', 'tpo_vei']],
    left_on='vehicle',
    right_on='tpo_vei',
    how='left'
).drop('tpo_vei', axis=1)

fat_crr = fat_crr.merge(
    dim_cli_db[['srk_cli', 'id_cli']],
    left_on='customer_id',
    right_on='id_cli',
    how='left'
).drop('id_cli', axis=1)

fat_crr = fat_crr.merge(
    dim_pag_db[['srk_pag', 'mtd_pag']],
    left_on='payment',
    right_on='mtd_pag',
    how='left'
).drop('mtd_pag', axis=1)

print(f'Fato Corridas preparado: {len(fat_crr)} registros')

print(f'\nVerificando valores NULL nas chaves estrangeiras...')
print(f"srk_tmp NULL: {fat_crr['srk_tmp'].isna().sum()}")
print(f"srk_loc_ori NULL: {fat_crr['srk_loc_ori'].isna().sum()}")
print(f"srk_loc_dst NULL: {fat_crr['srk_loc_dst'].isna().sum()}")
print(f"srk_vei NULL: {fat_crr['srk_vei'].isna().sum()}")
print(f"srk_cli NULL: {fat_crr['srk_cli'].isna().sum()}")
print(f"srk_pag NULL: {fat_crr['srk_pag'].isna().sum()}")

registros_antes = len(fat_crr)
fat_crr = fat_crr.dropna(subset=['srk_tmp', 'srk_loc_ori', 'srk_loc_dst', 'srk_vei', 'srk_cli', 'srk_pag'])
registros_depois = len(fat_crr)

print(f'\nRegistros removidos por falta de chaves: {registros_antes - registros_depois}')
print(f'Registros validos: {registros_depois}')


Fato Corridas preparado: 148767 registros

Verificando valores NULL nas chaves estrangeiras...
srk_tmp NULL: 0
srk_loc_ori NULL: 0
srk_loc_dst NULL: 0
srk_vei NULL: 0
srk_cli NULL: 0
srk_pag NULL: 47592

Registros removidos por falta de chaves: 47592
Registros validos: 101175


## 14. Preparar Colunas do Fato

Selecionar e renomear colunas para o fato.

In [144]:
fat_crr_final = fat_crr[[
    'srk_tmp', 'srk_loc_ori', 'srk_loc_dst', 'srk_vei', 'srk_cli', 'srk_pag',
    'distance', 'vtat', 'value', 'value_per_km',
    'driver_rating', 'customer_rating', 'status', 'distance_category',
    'pickup_region', 'drop_region', 'id'
]].copy()

fat_crr_final.rename(columns={
    'distance': 'dtc_km',
    'vtat': 'drc_min',
    'value': 'vlr_crr',
    'value_per_km': 'vlr_por_km',
    'driver_rating': 'avl_mtr',
    'customer_rating': 'avl_cli',
    'status': 'stt_crr',
    'distance_category': 'ctg_dtc',
    'pickup_region': 'pck_reg', 
    'drop_region': 'drp_reg',
    'id': 'id_bok_org'
}, inplace=True)

fat_crr_final['ver_crr_cpl'] = fat_crr_final['stt_crr'] == 'Completed'
fat_crr_final['ver_crr_cnl'] = fat_crr_final['stt_crr'].isin(['Cancelled by Driver', 'Cancelled by Customer'])
fat_crr_final['ver_rta_itr_rgl'] = fat_crr_final['pck_reg'] != fat_crr_final['drp_reg']

fat_crr_final['dat_hra_col'] = pd.Timestamp.now()

fat_crr_final.drop(['pck_reg', 'drp_reg'], axis=1, inplace=True)

print(f'Fato final preparado: {len(fat_crr_final)} registros')
print(f'Colunas: {list(fat_crr_final.columns)}')

Fato final preparado: 101175 registros
Colunas: ['srk_tmp', 'srk_loc_ori', 'srk_loc_dst', 'srk_vei', 'srk_cli', 'srk_pag', 'dtc_km', 'drc_min', 'vlr_crr', 'vlr_por_km', 'avl_mtr', 'avl_cli', 'stt_crr', 'ctg_dtc', 'id_bok_org', 'ver_crr_cpl', 'ver_crr_cnl', 'ver_rta_itr_rgl', 'dat_hra_col']


## 15. Inserir Fato no Banco

Carregar tabela fato no schema DW.

In [145]:
fat_crr_final.to_sql('fat_crr', engine, schema='dw', if_exists='append', index=False)

print(f'fat_crr: {len(fat_crr_final)} registros inseridos')
print('\nTabela fato carregada com sucesso!')

fat_crr: 101175 registros inseridos

Tabela fato carregada com sucesso!


## 16. Diagnostico Pos-ETL

Verificar contagem de registros em todas as tabelas.

In [146]:
cur.execute("SELECT COUNT(*) FROM dw.dim_tmp")
print(f'dim_tmp: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.dim_loc")
print(f'dim_loc: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.dim_vei")
print(f'dim_vei: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.dim_cli")
print(f'dim_cli: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.dim_pag")
print(f'dim_pag: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.fat_crr")
print(f'fat_crr: {cur.fetchone()[0]:,} registros')

print('\nETL Silver → Gold concluido com sucesso!')

dim_tmp: 365 registros
dim_loc: 176 registros
dim_vei: 7 registros
dim_cli: 147,580 registros
dim_pag: 5 registros
fat_crr: 101,175 registros

ETL Silver → Gold concluido com sucesso!


## 17. Fechar Conexoes

Encerrar conexoes com o banco de dados.

In [147]:
cur.close()
conn.close()
engine.dispose()

print('Conexoes fechadas.')

Conexoes fechadas.
